In [1]:
print("hi")

hi


In [2]:
from git import Repo
import os

from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser

from langchain_text_splitters import RecursiveCharacterTextSplitter,Language
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_groq import ChatGroq

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

e:\Courses_Projects\AIProjects\GenerativeAI\RealtimeSourceCodeAnalyzer\Realtime-Source-Code-Analyzer\Code_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
%pwd

'e:\\Courses_Projects\\AIProjects\\GenerativeAI\\RealtimeSourceCodeAnalyzer\\Realtime-Source-Code-Analyzer\\research'

In [ ]:
!mkdir test_repo

In [4]:
repo_path = "test_repo/"

In [ ]:

repo = Repo.clone_from("https://github.com/entbappy/End-to-end-Medical-Chatbot-Generative-AI", to_path=repo_path)

In [5]:
loader = GenericLoader.from_filesystem(repo_path,
                                        glob = "**/*",
                                       suffixes=[".py"],
                                       parser = LanguageParser(language=Language.PYTHON, parser_threshold=500)
)

In [6]:
documents = loader.load()


In [7]:
documents

[Document(metadata={'source': 'test_repo\\app.py', 'language': <Language.PYTHON: 'python'>}, page_content='from flask  import Flask,render_template,jsonify,request\nfrom src.helper import download_huggingface_embeddings\nfrom langchain_pinecone import PineconeVectorStore\n\nfrom transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline\nfrom langchain.llms import HuggingFacePipeline\n\nfrom langchain.chains import create_retrieval_chain\nfrom langchain.chains.combine_documents import create_stuff_documents_chain\nfrom langchain_core.prompts import ChatPromptTemplate\n\nfrom dotenv import load_dotenv\n\nfrom src.prompt import *\nimport os\n\napp =Flask(__name__)\n\nload_dotenv()\n\n# OPENAI_API_KEY= os.getenv("OPENAI_API_KEY")\nHF_API_KEY= os.getenv("HF_API_KEY")\nPINECONE_API_KEY= os.getenv("PINECONE_API_KEY")\n\n# os.environ["OPENAI_API_KEY"]=OPENAI_API_KEY\nos.environ["HF_API_KEY"]= HF_API_KEY\nos.environ["PINECONE_API_KEY"]=PINECONE_API_KEY\n\nembeddings = download_huggingf

In [8]:
documents_splitter = RecursiveCharacterTextSplitter.from_language(language = Language.PYTHON,
                                                             chunk_size = 500,
                                                             chunk_overlap = 20)

In [9]:
texts = documents_splitter.split_documents(documents)


In [10]:
texts

[Document(metadata={'source': 'test_repo\\app.py', 'language': <Language.PYTHON: 'python'>}, page_content='from flask  import Flask,render_template,jsonify,request\nfrom src.helper import download_huggingface_embeddings\nfrom langchain_pinecone import PineconeVectorStore\n\nfrom transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline\nfrom langchain.llms import HuggingFacePipeline\n\nfrom langchain.chains import create_retrieval_chain\nfrom langchain.chains.combine_documents import create_stuff_documents_chain\nfrom langchain_core.prompts import ChatPromptTemplate\n\nfrom dotenv import load_dotenv'),
 Document(metadata={'source': 'test_repo\\app.py', 'language': <Language.PYTHON: 'python'>}, page_content='from src.prompt import *\nimport os\n\napp =Flask(__name__)\n\nload_dotenv()\n\n# OPENAI_API_KEY= os.getenv("OPENAI_API_KEY")\nHF_API_KEY= os.getenv("HF_API_KEY")\nPINECONE_API_KEY= os.getenv("PINECONE_API_KEY")\n\n# os.environ["OPENAI_API_KEY"]=OPENAI_API_KEY\nos.environ["

In [11]:
len(texts)

15

In [12]:
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY=os.environ.get('GROQ_API_KEY')

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

llm = ChatGroq(
    model="llama-3.3-70b-versatile"
    # "llama3-8b-8192"
)

In [13]:
# HuggingFace Embeddings 

# embeddings = HuggingFaceEmbeddings()
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


C:\Users\Rahana\AppData\Local\Temp\ipykernel_14516\12332380.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2703.46it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
#Vector DB (Chroma)
vectordb = Chroma.from_documents(texts, embedding=embeddings, persist_directory='./db')

In [15]:
retriever = vectordb.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 8}
)

In [16]:
prompt = ChatPromptTemplate.from_template("""
You are an expert code assistant.

Use ONLY the context below to answer the question.

Context:
{context}

Question:
{question}

Answer clearly and technically.
""")

In [17]:
#  Helper function
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


In [18]:
#LCEL 
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)


In [19]:
question = "what is download_hugging_face_embeddings function doing?"
print(rag_chain.invoke(question))


The `download_huggingface_embeddings` function is designed to download and return a Hugging Face Embeddings model. 

Technically, it performs the following operations:

1. Specifies the `model_name` as "sentence-transformers/all-MiniLM-L6-v2", which is a pre-trained model available on the Hugging Face model hub.
2. Initializes an instance of the `HuggingFaceEmbeddings` class, passing the `model_name` as an argument to it.
3. Returns the initialized `HuggingFaceEmbeddings` instance, which can be used for embedding text data.

In essence, this function sets up a specific pre-trained language model for generating embeddings, which can be utilized in downstream natural language processing tasks, such as semantic search, text classification, or question-answering. 

The returned `HuggingFaceEmbeddings` instance is stored in the `embeddings` variable, making it available for use in other parts of the application.


In [20]:
question = "what is text_split function doing?"
print(rag_chain.invoke(question))


**Text Split Function Analysis**

The `text_split` function is designed to split a list of documents (`minimal_docs`) into smaller chunks of text. This is achieved by utilizing the `RecursiveCharacterTextSplitter` class from the `langchain.text_splitter` module.

**Function Breakdown**

1. **Text Splitter Initialization**: An instance of `RecursiveCharacterTextSplitter` is created with the following parameters:
   - `chunk_size`: 500 (the maximum size of each chunk in characters)
   - `chunk_overlap`: 20 (the overlap between consecutive chunks in characters)

2. **Text Splitting**: The `split_documents` method of the `text_splitter` instance is called, passing the `minimal_docs` list as an argument. This method splits the input documents into smaller chunks based on the specified `chunk_size` and `chunk_overlap`.

3. **Return Value**: The resulting text chunks are returned by the `text_split` function.

**Purpose**

The primary purpose of the `text_split` function is to divide large do

In [21]:
question = "what is the use of download_hugging_face_embeddings?"
print(rag_chain.invoke(question))


The `download_huggingface_embeddings` function is used to download and return a HuggingFace Embeddings model, specifically the "sentence-transformers/all-MiniLM-L6-v2" model. 

This function utilizes the `HuggingFaceEmbeddings` class from the `langchain.embeddings` module to create an instance of the model with the specified `model_name`. The purpose of this model is to generate embeddings, which are dense vector representations of text, that can be used for various natural language processing tasks such as text similarity measurement, clustering, and information retrieval.

In the context of the provided code, the `download_huggingface_embeddings` function is used to initialize the `embeddings` variable, which is then used to create a `PineconeVectorStore` instance. This instance is used to store and manage vector embeddings of documents, enabling efficient similarity searches and other operations. 

Technically, the `download_huggingface_embeddings` function serves as a utility to fa

In [22]:
question = "what chat route does?"
print(rag_chain.invoke(question))

The `/get` route, not the `/` route, is the chat route. 

Technically, the `/get` route handles both GET and POST methods. When a request is made to this route, it:

1. Retrieves the message from the request form data using `request.form["msg"]`.
2. Prints the input message.
3. Invokes the `rag_chain` with the input message and stores the response.
4. Prints the response from the `rag_chain`.
5. Returns the answer from the response as a string.

In essence, the `/get` route is responsible for handling user input, querying the chat model, and returning the response to the user. 

Here is the specific code for the chat route:
```python
@app.route("/get", methods=["GET", "POST"])
def chat():
    msg = request.form["msg"]
    input = msg
    print(input)
    response = rag_chain.invoke({"input": msg})
    print("Response :", response["answer"])
    return str(response["answer"])
```


In [ ]:
# prompt = ChatPromptTemplate.from_template("""
# You are an expert code assistant.

# Use ONLY the context below to answer the question.

# Context:
# {context}

# Question:
# {question}

# Answer clearly and technically.
# """)

In [ ]:
prompt = ChatPromptTemplate.from_template("""
You are a senior software engineer.

Answer ONLY from the context.

If unsure, say "Not found in codebase".

Context:
{context}

Question:
{question}

Provide:
- Explanation
- Code behavior
- Purpose
""")

In [ ]:
# # Helper function
# def format_docs(docs):
#     return "\n\n".join(doc.page_content for doc in docs)

# from langchain_core.runnables import RunnableLambda

# def format_docs(docs):
#     return "\n\n".join([d.page_content for d in docs])

# format_docs = RunnableLambda(format_docs)


from langchain_core.runnables import RunnableLambda

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

format_docs = RunnableLambda(lambda docs: "\n\n".join(d.page_content for d in docs))

In [ ]:
# #LCEL RAG

# rag_chain = (
#     {
#         "context": retriever | format_docs,
#         "question": RunnablePassthrough()
#     }
#     | prompt
#     | llm
#     | StrOutputParser()
# )




In [ ]:
rag_chain = (
    RunnablePassthrough.assign(
        context=lambda x: format_docs.invoke(retriever.invoke(x["question"]))
    )
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

In [ ]:
store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

In [ ]:
rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)

In [ ]:
question = "what is download_hugging_face_embeddings function doing?"

response = rag_chain.invoke(
    {"question": question},
    config={"configurable": {"session_id": "chat_1"}}
)

print(response)

**Explanation**: 
The `download_huggingface_embeddings` function is designed to download and return a HuggingFace Embeddings model. This model is specifically used for generating embeddings, which are vector representations of text data.

**Code Behavior**: 
The function downloads the "sentence-transformers/all-MiniLM-L6-v2" model from HuggingFace and creates an instance of `HuggingFaceEmbeddings` with this model. The `model_name` variable stores the name of the model being downloaded, and the `embeddings` variable holds the instance of the downloaded model.

**Purpose**: 
The purpose of this function is to provide a way to easily download and use a pre-trained HuggingFace Embeddings model for generating embeddings. This is likely used in a larger application for tasks such as text similarity search, clustering, or other natural language processing tasks that rely on vector representations of text data.

Example code snippet from the context:
```python
def download_huggingface_embeddin

In [ ]:
question = "what is text_split function doing?"

response = rag_chain.invoke(
    {"question": question},
    config={"configurable": {"session_id": "chat_2"}}
)

print(response)

**Explanation**: The `text_split` function is designed to split a given set of documents (`minimal_docs`) into smaller chunks of text. This is achieved using a `RecursiveCharacterTextSplitter` object, which is configured with a `chunk_size` of 500 characters and a `chunk_overlap` of 20 characters.

**Code behavior**: The function creates a `RecursiveCharacterTextSplitter` object with the specified configuration, then calls the `split_documents` method on this object, passing in the `minimal_docs` parameter. The resulting text chunks are stored in the `text_chunk` variable and returned by the function.

**Purpose**: The purpose of the `text_split` function is to divide large documents into smaller, more manageable pieces of text, likely for further processing or analysis, such as embedding generation or question-answering tasks.


In [ ]:
question = "what is download_hugging_face_embeddings function doing?"
print(rag_chain.invoke(question))

ValueError: Missing keys ['session_id'] in config['configurable'] Expected keys are ['session_id'].When using via .invoke() or .stream(), pass in a config; e.g., chain.invoke({'question': 'foo'}, {'configurable': {'session_id': '[your-value-here]'}})

In [ ]:
question = "what is text_split function doing?"
print(rag_chain.invoke(question))

The `text_split` function is splitting input text documents into smaller chunks using the `RecursiveCharacterTextSplitter` class. It takes `minimal_docs` as input and returns a list of text chunks, with each chunk having a maximum size of 500 characters and an overlap of 20 characters between adjacent chunks. This is likely used for text processing or embedding tasks, such as preparing input for a language model.


In [ ]:
question = "what is the use of download_hugging_face_embeddings?"
print(rag_chain.invoke(question))

The `download_huggingface_embeddings` function is used to download and return the HuggingFace Embeddings model, specifically the "sentence-transformers/all-MiniLM-L6-v2" model. 

Technically, this function initializes a `HuggingFaceEmbeddings` object with the specified model name and returns it, allowing for the utilization of the downloaded model in subsequent operations, such as generating embeddings for text data. 

In the context provided, the embeddings generated by this model are likely used for indexing and searching purposes, specifically with the Pinecone vector database, to enable efficient similarity searches and information retrieval.


In [ ]:
from git import Repo



from langchain_text_splitters import Language, RecursiveCharacterTextSplitter

from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser
from langchain_community.vectorstores import Chroma

import os
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_groq import ChatGroq
from langchain_community.memory import ConversationSummaryMemory
from langchain_community.chains import ConversationalRetrievalChain

e:\Courses_Projects\AIProjects\GenerativeAI\RealtimeSourceCodeAnalyzer\Realtime-Source-Code-Analyzer\Code_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: cannot import name 'ConversationSummaryMemory' from 'langchain_community.memory' (e:\Courses_Projects\AIProjects\GenerativeAI\RealtimeSourceCodeAnalyzer\Realtime-Source-Code-Analyzer\Code_env\lib\site-packages\langchain_community\memory\__init__.py)

In [ ]:
from langchain.memory import ConversationSummaryMemory
from langchain.chains import ConversationalRetrievalChain


ModuleNotFoundError: No module named 'langchain.memory'

In [ ]:
import langchain
print(langchain.__version__)

1.2.15


In [ ]:
import sys
print(sys.executable)

e:\Courses_Projects\AIProjects\GenerativeAI\RealtimeSourceCodeAnalyzer\Realtime-Source-Code-Analyzer\Code_env\python.exe


In [ ]:
!pip show langchain

Name: langchain
Version: 1.2.15
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: e:\courses_projects\aiprojects\generativeai\realtimesourcecodeanalyzer\realtime-source-code-analyzer\code_env\lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 


In [ ]:
from langchain.chains import ConversationalRetrievalChain
print("OK")

ModuleNotFoundError: No module named 'langchain.chains'

In [ ]:
import os
from git import Repo
from dotenv import load_dotenv

from langchain_text_splitters import Language, RecursiveCharacterTextSplitter

from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser
from langchain_community.vectorstores import Chroma

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

# from langchain.chains import ConversationalRetrievalChain
from langchain_community.chains import ConversationalRetrievalChain
from langchain_community.memory import ConversationSummaryMemory
# from langchain.memory import ConversationBufferMemory

ImportError: cannot import name 'ConversationalRetrievalChain' from 'langchain_community.chains' (e:\Courses_Projects\AIProjects\GenerativeAI\RealtimeSourceCodeAnalyzer\Realtime-Source-Code-Analyzer\Code_env\lib\site-packages\langchain_community\chains\__init__.py)

In [ ]:
import os
from git import Repo
from dotenv import load_dotenv

from langchain_text_splitters  import Language, RecursiveCharacterTextSplitter

from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser
from langchain_community.vectorstores import Chroma

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

from langchain_community.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain


ImportError: cannot import name 'ConversationBufferMemory' from 'langchain_community.memory' (e:\Courses_Projects\AIProjects\GenerativeAI\RealtimeSourceCodeAnalyzer\Realtime-Source-Code-Analyzer\Code_env\lib\site-packages\langchain_community\memory\__init__.py)

In [ ]:
from langchain.memory.buffer import ConversationBufferMemory

ModuleNotFoundError: No module named 'langchain.memory'

In [ ]:
import os
from git import Repo
from dotenv import load_dotenv

from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import Language

from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_groq import ChatGroq

from langchain.memory import ConversationSummaryMemory
from langchain.chains import ConversationalRetrievalChain


e:\Courses_Projects\AIProjects\GenerativeAI\RealtimeSourceCodeAnalyzer\Realtime-Source-Code-Analyzer\Code_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'langchain.memory'

In [ ]:
from langchain_core.memory import ConversationSummaryMemory

ModuleNotFoundError: No module named 'langchain_core.memory'

In [ ]:
from langchain_community.memory import ConversationSummaryMemory

ImportError: cannot import name 'ConversationSummaryMemory' from 'langchain_community.memory' (e:\Courses_Projects\AIProjects\GenerativeAI\RealtimeSourceCodeAnalyzer\Realtime-Source-Code-Analyzer\Code_env\lib\site-packages\langchain_community\memory\__init__.py)

In [ ]:
# from git import Repo
# from langchain.text_splitter import Language
# from langchain.document_loaders.generic import GenericLoader
# from langchain.document_loaders.parsers import LanguageParser
# from langchain.text_splitter import RecursiveCharacterTextSplitter
# from langchain.embeddings import OpenAIEmbeddings
# from langchain.vectorstores import Chroma
# from langchain.chat_models import ChatOpenAI
# from langchain.memory import ConversationSummaryMemory
# from langchain.chains import ConversationalRetrievalChain
# import os

ModuleNotFoundError: No module named 'langchain.text_splitter'

In [ ]:
import os
from dotenv import load_dotenv

from git import Repo

# from langchain_community.document_loaders import GenericLoader
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_community.document_loaders.parsers import LanguageParser

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import Language

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

from langchain_groq import ChatGroq

from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationSummaryMemory

ModuleNotFoundError: No module named 'dotenv'

In [ ]:
# from git import Repo
# from langchain_text_splitters.base import Language
# from langchain_community.document_loaders.generic import GenericLoader
# from langchain_community.document_loaders.parsers import LanguageParser
# from langchain_text_splitters import RecursiveCharacterTextSplitter
# # from langchain.embeddings import OpenAIEmbeddings
# from langchain_community.vectorstores import Chroma
# # from langchain.chat_models import ChatOpenAI
# from langchain_groq import ChatGroq

# from langchain.memory import ConversationSummaryMemory
# from langchain.chains import ConversationalRetrievalChain

# import os

ModuleNotFoundError: No module named 'langchain.memory'

In [ ]:
# pip install langchain==0.2.11 langchain-community==0.2.11 langchain-core==0.2.11  langchain-text-splitters==0.2.4 langchain-groq

Clone the Github Repo

In [ ]:
!mkdir -p test_repo

In [ ]:
repo_path = 'test_repo'

In [ ]:

repo = Repo.clone_from("https://github.com/RahanaPA/End-to-End-Medical-Chatbot", to_path=repo_path)

In [ ]:
%pwd

'e:\\Courses_Projects\\AIProjects\\GenerativeAI\\RealtimeSourceCodeAnalyzer\\Realtime-Source-Code-Analyzer\\research'

In [ ]:
loader = GenericLoader.from_filesystem(repo_path,
                                       glob = "**/*",
                                       suffixes=[".py"],
                                       parser = LanguageParser(language=Language.PYTHON,parser_threshold=500)
                                       )

In [ ]:
documents = loader.load()

In [ ]:
documents

[Document(page_content='from flask  import Flask,render_template,jsonify,request\nfrom src.helper import download_huggingface_embeddings\nfrom langchain_pinecone import PineconeVectorStore\n\nfrom transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline\nfrom langchain.llms import HuggingFacePipeline\n\nfrom langchain.chains import create_retrieval_chain\nfrom langchain.chains.combine_documents import create_stuff_documents_chain\nfrom langchain_core.prompts import ChatPromptTemplate\n\nfrom dotenv import load_dotenv\n\nfrom src.prompt import *\nimport os\n\napp =Flask(__name__)\n\nload_dotenv()\n\n# OPENAI_API_KEY= os.getenv("OPENAI_API_KEY")\nHF_API_KEY= os.getenv("HF_API_KEY")\nPINECONE_API_KEY= os.getenv("PINECONE_API_KEY")\n\n# os.environ["OPENAI_API_KEY"]=OPENAI_API_KEY\nos.environ["HF_API_KEY"]= HF_API_KEY\nos.environ["PINECONE_API_KEY"]=PINECONE_API_KEY\n\nembeddings = download_huggingface_embeddings()\n\nindex_name = "medical-chatbot"\n#Embed each chunk and upsert th

In [ ]:
len(documents)

6

Split the documnets into chunks

In [ ]:
document_splitter = RecursiveCharacterTextSplitter.from_language(language= Language.PYTHON,
                                                                 chunk_size =500,
                                                                 chunk_overlap = 20)

In [ ]:
texts = document_splitter.split_documents(documents)

In [ ]:
texts

[Document(page_content='from flask  import Flask,render_template,jsonify,request\nfrom src.helper import download_huggingface_embeddings\nfrom langchain_pinecone import PineconeVectorStore\n\nfrom transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline\nfrom langchain.llms import HuggingFacePipeline\n\nfrom langchain.chains import create_retrieval_chain\nfrom langchain.chains.combine_documents import create_stuff_documents_chain\nfrom langchain_core.prompts import ChatPromptTemplate\n\nfrom dotenv import load_dotenv', metadata={'source': 'test_repo\\app.py', 'language': <Language.PYTHON: 'python'>}),
 Document(page_content='from src.prompt import *\nimport os\n\napp =Flask(__name__)\n\nload_dotenv()\n\n# OPENAI_API_KEY= os.getenv("OPENAI_API_KEY")\nHF_API_KEY= os.getenv("HF_API_KEY")\nPINECONE_API_KEY= os.getenv("PINECONE_API_KEY")\n\n# os.environ["OPENAI_API_KEY"]=OPENAI_API_KEY\nos.environ["HF_API_KEY"]= HF_API_KEY\nos.environ["PINECONE_API_KEY"]=PINECONE_API_KEY\n\nembedd

In [ ]:
len(texts)

15

Download the Embeddings

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings()

ImportError: cannot import name 'PydanticDeprecationWarning' from 'pydantic' (e:\Courses_Projects\AIProjects\GenerativeAI\RealtimeSourceCodeAnalyzer\Realtime-Source-Code-Analyzer\Code_env\lib\site-packages\pydantic\__init__.cp310-win_amd64.pyd)